[English](https://colab.research.google.com/github/kanoyo-git/rvc-rs/blob/main/NVC-Studio.ipynb) | [Русский](https://colab.research.google.com/github/kanoyo-git/rvc-rs/blob/main/NVC-Studio.ru.ipynb)

# NVC Studio в Google Colab

NVC Studio — реализация RVC на Rust: конвертация голоса, каверы, разделение вокала и обучение моделей в одном интерфейсе в браузере.

Блокнот скачивает сборку размером около 10 МБ, которая работает на PyTorch, уже установленном в Colab. Веса моделей заранее не скачиваются: когда вы нажимаете «Конвертировать», «Кавер», «Разделить» или «Обучить», Studio загружает только нужные этой задаче файлы и показывает прогресс.

Выберите среду с GPU (*Среда выполнения → Сменить среду выполнения → T4 GPU*) и запустите ячейки 0, 1 и 5. Ячейки 2–4 необязательны: голосовые модели с Google Drive, обучение с сохранением на Drive и TensorBoard.

**Туннель не работает?** Раздел *Без интерфейса* в конце делает всё то же, что Studio, прямо в ячейках.

In [ ]:
# @title 0 · Настройки
from pathlib import Path

#@markdown #### Сборка
ARCHIVE_URL = "https://github.com/kanoyo-git/rvc-rs/releases/download/v0.1.0/rvc-rs-colab-linux-x86_64-torch2141-cu130.tar.gz"  # @param {type:"string"}
ARCHIVE_SHA256 = "d9a92dced25fb2a3e9b6fae1b33e7df45e4223bc49fee9fa01c97cbcc9f4524c"  # @param {type:"string"}
PROJECT_DIR = "/content/nvc-studio"  # @param {type:"string"}

#@markdown #### Google Drive
USE_DRIVE_MODELS = False  # @param {type:"boolean"}
DRIVE_MODELS_PATH = "Models"  # @param {type:"string"}
USE_DRIVE_TRAINING = False  # @param {type:"boolean"}
DRIVE_TRAINING_PATH = "NVC-training"  # @param {type:"string"}
DRIVE_SYNC_MINUTES = 10  # @param {type:"integer"}

#@markdown #### Доступ
#@markdown gradio: встроенная ссылка Gradio. cloudflared и localtunnel: сторонние туннели. Везде нужен логин rvc и пароль.
ACCESS = "gradio"  # @param ["gradio", "cloudflared", "localtunnel"]
SHARE_PASSWORD = ""  # @param {type:"string"}
GUI_PORT = 7865  # @param {type:"integer"}

PROJECT = Path(PROJECT_DIR)
INPUT_DIR = Path("/content/nvc-input")
OUTPUT_DIR = Path("/content/nvc-output")
print("project:", PROJECT)
print("access:", ACCESS)


In [ ]:
# @title 1 · Установка NVC Studio
import hashlib, os, platform, re, shlex, shutil, subprocess, tarfile, tempfile, time
from importlib.metadata import PackageNotFoundError, distribution
from pathlib import Path

import requests
from IPython.display import Audio, Pretty, display
from tqdm.auto import tqdm

if platform.machine() != "x86_64":
    raise RuntimeError("Linux x86_64")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Среда выполнения -> Сменить среду выполнения -> T4 GPU")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     check=True, capture_output=True, text=True).stdout.strip())

EXPECTED_TORCH = "2.14.1+cu130"
try:
    torch_dist = distribution("torch")
except PackageNotFoundError as error:
    raise RuntimeError("В этой среде Colab нет предустановленного PyTorch") from error
if torch_dist.version != EXPECTED_TORCH:
    raise RuntimeError("Сборка рассчитана на Colab с PyTorch {expected}, а в этой среде PyTorch {found}. Новая сборка: https://github.com/kanoyo-git/rvc-rs/releases".format(expected=EXPECTED_TORCH, found=torch_dist.version))
torch_lib = Path(torch_dist.locate_file("torch/lib")).resolve()
if not (torch_lib / "libtorch_cuda.so").is_file():
    raise RuntimeError("У предустановленного PyTorch нет libtorch_cuda.so")
if not (shutil.which("ffmpeg") and shutil.which("ffprobe")):
    raise RuntimeError("В среде нет ffmpeg и ffprobe")
lib_dirs = [str(torch_lib), *map(str, sorted((torch_lib.parent.parent / "nvidia").glob("*/lib")))]
RVC_ENV = os.environ.copy()
RVC_ENV["LD_LIBRARY_PATH"] = os.pathsep.join(lib_dirs + [RVC_ENV.get("LD_LIBRARY_PATH", "")]).rstrip(os.pathsep)
print("PyTorch", torch_dist.version)

def _download(url, destination, checksum):
    digest = hashlib.sha256()
    with requests.get(url, stream=True, timeout=(20, 120)) as response:
        response.raise_for_status()
        size = int(response.headers.get("Content-Length", 0)) or None
        with open(destination, "wb") as output, tqdm(total=size, unit="B", unit_scale=True, desc=destination.name) as bar:
            for chunk in response.iter_content(1 << 20):
                output.write(chunk)
                digest.update(chunk)
                bar.update(len(chunk))
    if checksum and digest.hexdigest() != checksum.strip().lower():
        raise RuntimeError("SHA-256 архива не совпадает")

PROJECT.mkdir(parents=True, exist_ok=True)
marker = PROJECT / ".release"
if not (PROJECT / "rvc").is_file() or not marker.is_file() or marker.read_text().strip() != ARCHIVE_SHA256.strip():
    print("Устанавливаю NVC Studio")
    with tempfile.TemporaryDirectory(dir="/content") as staging:
        staging = Path(staging)
        archive = staging / "nvc-studio.tar.gz"
        _download(ARCHIVE_URL.strip(), archive, ARCHIVE_SHA256)
        with tarfile.open(archive, "r:gz") as release:
            release.extractall(staging, filter="data")
        unpacked = staging / "rvc-rs-colab"
        # Program files are replaced; voices, experiments and downloaded weights stay.
        shutil.copytree(unpacked, PROJECT, dirs_exist_ok=True)
    marker.write_text(ARCHIVE_SHA256.strip())
for folder in ("assets/weights", "assets/indices", "logs"):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RVC = PROJECT / "rvc"
WEIGHTS = PROJECT / "assets" / "weights"
INDICES = PROJECT / "assets" / "indices"

# Lines that only move a counter are shown on one updating line.
PROGRESS = re.compile(r"(\[\d+/\d+\] .*?)\s*\d+/\d+$|(Epoch \d+/\d+) · batch \d+/\d+$|\d+/\d+ - ()|(Processed) \d+/\d+ ")

def rvc(*args, check=True, env=None):
    """Run the rvc CLI; download progress is shown on one updating line."""
    command = [str(RVC), "--root", str(PROJECT), *map(str, args)]
    print("$ rvc " + " ".join(shlex.quote(str(a)) for a in args))
    process = subprocess.Popen(command, cwd=PROJECT, env=env or RVC_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    progress, label = None, None
    try:
        for line in process.stdout:
            line = line.rstrip()
            stage = PROGRESS.match(line)
            if line.startswith(("Downloading ", "Checking ")) or stage:
                name = next((group for group in stage.groups() if group is not None), "") if stage else line.split(" · ")[0].split(" ", 1)[-1]
                if progress is None or name != label:
                    progress, label = display(Pretty(line), display_id=True), name
                else:
                    progress.update(Pretty(line))
            else:
                progress = None
                print(line, flush=True)
    except KeyboardInterrupt:
        process.terminate()
        raise
    code = process.wait()
    if check and code:
        raise RuntimeError("rvc exited with code %d" % code)
    return code

def models():
    return sorted(p.name for p in WEIGHTS.iterdir() if p.suffix.lower() == ".pth")

def model_path(name):
    name = str(name).strip()
    for candidate in (Path(name), WEIGHTS / name, WEIGHTS / (name + ".pth")):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Голосовая модель не найдена: {name}. Есть: {models}".format(name=name, models=", ".join(models()) or "-"))

def index_args(model, choice, rate):
    """auto: rvc picks the voice's own index, or converts without one."""
    choice = str(choice).strip()
    if choice.lower() == "none" or not rate:
        return ["--index-rate", 0]
    if choice and choice.lower() != "auto":
        return ["--index", choice, "--index-rate", rate]
    return ["--index-rate", rate]

def input_path(value):
    value = str(value).strip()
    path = Path(value) if value.startswith("/") else INPUT_DIR / value
    if not value or not path.exists():
        raise FileNotFoundError("Не найдено: {path}".format(path=path))
    return path

AUDIO_SUFFIXES = {".wav", ".flac", ".mp3", ".m4a", ".ogg", ".opus", ".aac", ".webm", ".mp4"}

def show_new(folder, since, limit=6):
    files = sorted((p for p in Path(folder).rglob("*") if p.is_file() and p.stat().st_mtime >= since), key=lambda p: p.name)
    if not files:
        print("новых файлов нет")
    for path in [p for p in files if p.suffix.lower() in AUDIO_SUFFIXES][:limit]:
        print(path.name)
        display(Audio(filename=str(path)))

rvc("--version")
print("NVC Studio установлена в", PROJECT)


In [ ]:
# @title 2 · Голосовые модели с Google Drive
# <DRIVE_MODELS_PATH>/<Имя модели>/<любое>.pth + <любое>.index -> assets/weights/<Имя модели>.pth и assets/indices/<Имя модели>_<любое>.index
# Ничего не копируется: Studio видит симлинки на Drive.
import re
from pathlib import Path

if "RVC" not in globals():
    raise RuntimeError("Сначала запустите ячейку 1")
if not USE_DRIVE_MODELS:
    print("выключено", "(USE_DRIVE_MODELS = False)")
else:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    given = Path(str(DRIVE_MODELS_PATH).strip().strip('"').rstrip("/") or "Models")
    root = given if given.is_absolute() else Path("/content/drive/MyDrive") / given
    if not root.is_dir():
        raise RuntimeError("Папка с моделями не найдена: {path}. Проверьте DRIVE_MODELS_PATH (путь от MyDrive). Для общей папки добавьте ярлык в MyDrive.".format(path=root))

    def link(source, target):
        if target.exists() and not target.is_symlink():
            return
        if target.is_symlink():
            target.unlink()
        target.symlink_to(source)
        print("ссылка", target.relative_to(PROJECT), "->", source)

    found = 0
    groups = {}
    for source in sorted(root.rglob("*")):
        if source.is_file() and source.suffix.lower() in (".pth", ".index"):
            relative = source.relative_to(root)
            group = relative.parts[0] if len(relative.parts) > 1 else source.stem
            groups.setdefault(group, []).append(source)
    for group, files in groups.items():
        weights = [p for p in files if p.suffix.lower() == ".pth"]
        for source in weights:
            name = group if len(weights) == 1 else group + "_" + source.stem
            link(source, WEIGHTS / (name + ".pth"))
            found += 1
        for source in (p for p in files if p.suffix.lower() == ".index"):
            link(source, INDICES / (group + "_" + re.sub(r"(?i)(^|_)trained_ivf", r"\1added_IVF", source.stem) + ".index"))
            found += 1
    if not found:
        print("нет файлов .pth и .index в", root)


In [ ]:
# @title 3 · Обучение на Google Drive
# Восстанавливает эксперименты (logs/<имя>) и обученные голоса из DRIVE_TRAINING_PATH и
# синхронизирует их, пока работает Studio. На Drive хранится только последняя полная пара G/D.
import os, re, shutil, time
from pathlib import Path

if "RVC" not in globals():
    raise RuntimeError("Сначала запустите ячейку 1")

def _copy_if_changed(src, dst):
    if dst.is_symlink():
        return 0
    if dst.is_file():
        a, b = src.stat(), dst.stat()
        if a.st_size == b.st_size and int(a.st_mtime) == int(b.st_mtime):
            return 0
    dst.parent.mkdir(parents=True, exist_ok=True)
    partial = dst.with_name(dst.name + ".part")
    shutil.copy2(src, partial)
    os.replace(partial, dst)
    return 1

def _latest_pair(folder, settled):
    steps = {}
    for path in folder.glob("[GD]_*.pth"):
        match = re.fullmatch(r"[GD]_(\d+)\.pth", path.name)
        if match and path.stat().st_mtime <= settled:
            steps.setdefault(int(match.group(1)), {})[path.name[0]] = path
    complete = [step for step, pair in steps.items() if len(pair) == 2]
    return set(steps[max(complete)].values()) if complete else set()

def sync_training_to_drive(settle=30):
    copied, fresh = 0, time.time() - settle
    for experiment in sorted(p for p in (PROJECT / "logs").iterdir() if p.is_dir() and p.name != "mute"):
        target = TRAINING_DRIVE / "logs" / experiment.name
        keep = _latest_pair(experiment, fresh)
        for src in experiment.rglob("*"):
            if not src.is_file() or src.suffix in (".part", ".tmp") or src.stat().st_mtime > fresh:
                continue
            if src.parent == experiment and re.fullmatch(r"[GD]_\d+\.pth", src.name) and src not in keep:
                continue
            copied += _copy_if_changed(src, target / src.relative_to(experiment))
        if keep and all((target / p.name).is_file() for p in keep):
            for old in target.glob("[GD]_*.pth"):
                if old.name not in {p.name for p in keep}:
                    old.unlink()
    for src in WEIGHTS.glob("*.pth"):
        if not src.is_symlink() and src.stat().st_mtime <= fresh:
            copied += _copy_if_changed(src, TRAINING_DRIVE / "weights" / src.name)
    for src in INDICES.glob("*.index"):
        if not src.is_symlink() and src.stat().st_mtime <= fresh:
            copied += _copy_if_changed(src, TRAINING_DRIVE / "indices" / src.name)
    return copied

def restore_training_from_drive():
    restored = 0
    for experiment in sorted(p for p in (TRAINING_DRIVE / "logs").glob("*") if p.is_dir()):
        for src in experiment.rglob("*"):
            if src.is_file() and src.suffix != ".part":
                restored += _copy_if_changed(src, PROJECT / "logs" / experiment.name / src.relative_to(experiment))
    for kind, folder in (("weights", WEIGHTS), ("indices", INDICES)):
        for src in (TRAINING_DRIVE / kind).glob("*"):
            if src.is_file():
                restored += _copy_if_changed(src, folder / src.name)
    return restored

if not USE_DRIVE_TRAINING:
    print("выключено", "(USE_DRIVE_TRAINING = False)")
else:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    given = Path(str(DRIVE_TRAINING_PATH).strip().strip('"').rstrip("/") or "NVC-training")
    TRAINING_DRIVE = given if given.is_absolute() else Path("/content/drive/MyDrive") / given
    for kind in ("logs", "weights", "indices"):
        (TRAINING_DRIVE / kind).mkdir(parents=True, exist_ok=True)
    print("восстановлено с Drive файлов: {n}".format(n=restore_training_from_drive()))
    names = sorted(p.name for p in (TRAINING_DRIVE / "logs").iterdir() if p.is_dir())
    print("эксперименты на Drive:", ", ".join(names)) if names else print("на Drive пока нет экспериментов:", TRAINING_DRIVE)


In [ ]:
# @title 4 · TensorBoard (необязательно)
TENSORBOARD_PORT = 6006  # @param {type:"integer"}
ipython = get_ipython()
ipython.run_line_magic("load_ext", "tensorboard")
ipython.run_line_magic("tensorboard", "--logdir %s --port %d --reload_interval 15" % (PROJECT / "logs", TENSORBOARD_PORT))


In [ ]:
# @title 5 · Запуск NVC Studio
import os, re, secrets, shutil, socket, subprocess, sys, threading, time
from pathlib import Path

if "RVC" not in globals():
    raise RuntimeError("Сначала запустите ячейку 1")
if USE_DRIVE_TRAINING and "sync_training_to_drive" not in globals():
    raise RuntimeError("USE_DRIVE_TRAINING: run cell 3")

CHILDREN = []
password = SHARE_PASSWORD.strip() or secrets.token_urlsafe(9)
server_env = dict(RVC_ENV, RVC_SHARE_PASSWORD=password)

def port_open(port):
    with socket.socket() as sock:
        sock.settimeout(0.4)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def pump(stream, found=None, pattern=None):
    for line in stream:
        sys.stdout.write(line)
        sys.stdout.flush()
        if found is not None and not found and pattern:
            match = re.search(pattern, line)
            if match:
                found.append(match.group(0))

def start(command, env=None, pattern=None):
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    CHILDREN.append(process)
    found = []
    threading.Thread(target=pump, args=(process.stdout, found, pattern), daemon=True).start()
    return process, found

def wait_for(found, process, timeout=120):
    deadline = time.time() + timeout
    while not found and time.time() < deadline and process.poll() is None:
        time.sleep(0.3)
    if not found:
        raise RuntimeError("Туннель не выдал адрес. Выберите другой ACCESS или используйте ячейки из раздела «Без интерфейса».")
    return found[0]

def fetch_tool(url, path):
    path = Path(path)
    if not path.is_file():
        subprocess.run(["curl", "-fsSL", "-o", str(path), url], check=True)
        path.chmod(0o755)
    return path

for process in CHILDREN:
    process.terminate()
if port_open(GUI_PORT):
    subprocess.run(["pkill", "-f", "rvc --root %s serve" % PROJECT])
    time.sleep(1)

serve = [str(RVC), "--root", str(PROJECT), "serve", "--port", str(GUI_PORT)]
if ACCESS == "gradio":
    frpc = fetch_tool("https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64", "/content/frpc_linux_amd64_v0.3")
    server, found = start(serve + ["--colab", "--share-client", str(frpc)], server_env, r"https://[a-z0-9-]+\.gradio\.live/\S*")
    url = wait_for(found, server, 90)
else:
    if ACCESS == "cloudflared":
        tool = fetch_tool("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
        tunnel, found = start([str(tool), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:%d" % GUI_PORT], None, r"https://[a-z0-9-]+\.trycloudflare\.com")
    else:
        tunnel, found = start(["npx", "--yes", "localtunnel", "--port", str(GUI_PORT)], None, r"https://\S+\.loca\.lt")
    url = wait_for(found, tunnel)
    server, _ = start(serve + ["--public-url", url], server_env)
deadline = time.time() + 60
while not port_open(GUI_PORT) and server.poll() is None and time.time() < deadline:
    time.sleep(0.3)

print()
print("=" * 60)
print("Откройте Studio" + ":", url)
print("Логин" + ": rvc   " + "пароль" + ":", password)
if ACCESS == "localtunnel":
    print("LocalTunnel сначала спросит пароль туннеля, это IP этой среды:", requests.get("https://loca.lt/mytunnelpassword", timeout=10).text.strip())
print("=" * 60)

if USE_DRIVE_TRAINING:
    def drive_sync_loop():
        while True:
            time.sleep(max(1, DRIVE_SYNC_MINUTES) * 60)
            try:
                copied = sync_training_to_drive()
                if copied:
                    print("[drive] synced %d file(s)" % copied)
            except Exception as error:
                print("[drive] sync failed:", error)
    threading.Thread(target=drive_sync_loop, daemon=True).start()

started = time.time()
try:
    while server.poll() is None:
        time.sleep(300)
        print("[heartbeat] %d min" % ((time.time() - started) // 60))
except KeyboardInterrupt:
    print("Останавливаю Studio и туннели")
finally:
    for process in reversed(CHILDREN):
        process.terminate()
    for process in CHILDREN:
        try:
            process.wait(timeout=5)
        except Exception:
            process.kill()
    if USE_DRIVE_TRAINING:
        print("Последняя синхронизация с Google Drive")
        print("[drive] synced %d file(s)" % sync_training_to_drive(settle=0))
    print("Остановлено. Ячейку можно запустить снова.")


## Без интерфейса

Эти ячейки для случая, когда туннель недоступен или удобнее работать в ячейках. Нужны ячейки 0 и 1 (и 2 для моделей с Drive). Каждая ячейка выполняет одну операцию Studio; недостающие веса скачиваются с прогрессом, как в Studio.

Файлы кладите в **/content/nvc-input** (ячейка A или панель «Файлы» слева). Результаты сохраняются в **/content/nvc-output**.

In [ ]:
# @title A · Загрузка файлов
import shutil
from pathlib import Path
from google.colab import files

print("Выберите файлы. Голосовые модели (.pth) попадут в assets/weights, индексы (.index) в assets/indices, остальное в /content/nvc-input.")
for name in files.upload(target_dir=str(INPUT_DIR)):
    path = INPUT_DIR / Path(name).name
    suffix = path.suffix.lower()
    if suffix == ".pth":
        shutil.move(path, WEIGHTS / path.name)
        print("model:", WEIGHTS / path.name)
    elif suffix == ".index":
        shutil.move(path, INDICES / path.name)
        print("index:", INDICES / path.name)
    else:
        print("input:", path)


In [ ]:
# @title B · Конвертация
#@markdown Голосовая модель: имя из assets/weights (ячейка A или 2) или полный путь. Вход: файл или папка в /content/nvc-input или полный путь.
MODEL = ""  # @param {type:"string"}
INPUT = ""  # @param {type:"string"}
PITCH = 0  # @param {type:"slider", min:-24, max:24, step:1}
F0_METHOD = "rmvpe"  # @param ["rmvpe", "nmvpe", "pm", "fcpe"]
#@markdown Индекс: auto выбирает индекс этой модели, none выключает его, или укажите путь.
INDEX = "auto"  # @param {type:"string"}
INDEX_RATE = 0.75  # @param {type:"slider", min:0, max:1, step:0.05}
PROTECT = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}
RMS_MIX_RATE = 1.0  # @param {type:"slider", min:0, max:1, step:0.05}
APERIODIC = 1.0  # @param {type:"slider", min:0, max:1, step:0.05}
LEVEL_GUARD = True  # @param {type:"boolean"}
VOICE_GATE = 0  # @param {type:"slider", min:0, max:30, step:1}
RESAMPLE_SR = 0  # @param {type:"integer"}
SPEAKER_ID = ""  # @param {type:"string"}
PRECISION = "auto"  # @param ["auto", "fp32", "fp16"]
FORMAT = "wav"  # @param ["wav", "flac", "mp3", "m4a"]
import time

model = model_path(MODEL)
source = input_path(INPUT)
target = OUTPUT_DIR / "convert" / model.stem
target.mkdir(parents=True, exist_ok=True)
args = ["infer", "--model", model, "--input", source, "--output", target, "--pitch", PITCH,
        "--f0-method", F0_METHOD, "--protect", PROTECT, "--rms-mix-rate", RMS_MIX_RATE,
        "--aperiodic", APERIODIC, "--voice-gate", VOICE_GATE, "--resample-sr", RESAMPLE_SR,
        "--precision", PRECISION, "--format", FORMAT, "--overwrite", "--device", "cuda:0"]
args += index_args(model, INDEX, INDEX_RATE)
if source.is_dir():
    args.append("--recursive")
if not LEVEL_GUARD:
    args.append("--no-level-guard")
if SPEAKER_ID.strip():
    args += ["--speaker-id", SPEAKER_ID.strip()]
started = time.time() - 1
rvc(*args)
show_new(target, started)


In [ ]:
# @title C · Кавер
#@markdown Песня (или вокальная дорожка с VOCALS_ONLY) делится на основной вокал, бэки и реверб, каждый сухой слой конвертируется, затем микс собирается заново.
MODEL = ""  # @param {type:"string"}
INPUT = ""  # @param {type:"string"}
PITCH = 0  # @param {type:"slider", min:-24, max:24, step:1}
INDEX = "auto"  # @param {type:"string"}
INDEX_RATE = 0.5  # @param {type:"slider", min:0, max:1, step:0.05}
BACKING = "convert"  # @param ["convert", "keep", "drop"]
DRY = False  # @param {type:"boolean"}
VOCAL_GAIN = 0.0  # @param {type:"slider", min:-12, max:12, step:0.5}
APERIODIC = 1.0  # @param {type:"slider", min:0, max:1, step:0.05}
VOICE_GATE = 0  # @param {type:"slider", min:0, max:30, step:1}
VOCALS_ONLY = False  # @param {type:"boolean"}
#@markdown С VOCALS_ONLY: минус, с которым смешать сконвертированный вокал (необязательно).
INSTRUMENTAL = ""  # @param {type:"string"}
SPEAKER_ID = ""  # @param {type:"string"}
DEVICE = "auto"  # @param ["auto", "cuda:0", "cpu"]
FORMAT = "flac"  # @param ["flac", "wav", "mp3", "m4a"]
import time

model = model_path(MODEL)
source = input_path(INPUT)
sources = sorted(p for p in source.rglob("*") if p.suffix.lower() in AUDIO_SUFFIXES) if source.is_dir() else [source]
target = OUTPUT_DIR / "cover" / model.stem
target.mkdir(parents=True, exist_ok=True)
args = ["cover", "--model", model, "--output", target, "--pitch", PITCH, "--backing", BACKING,
        "--vocal-gain", VOCAL_GAIN, "--aperiodic", APERIODIC, "--voice-gate", VOICE_GATE,
        "--device", DEVICE, "--format", FORMAT]
for path in sources:
    args += ["--input", path]
args += index_args(model, INDEX, INDEX_RATE)
if DRY:
    args.append("--dry")
if VOCALS_ONLY:
    args.append("--vocals-only")
    if INSTRUMENTAL.strip():
        args += ["--instrumental", input_path(INSTRUMENTAL)]
if SPEAKER_ID.strip():
    args += ["--speaker-id", SPEAKER_ID.strip()]
started = time.time() - 1
rvc(*args)
show_new(target, started)


In [ ]:
# @title D · Разделение вокала
INPUT = ""  # @param {type:"string"}
PRESET = "vocals-bs-roformer-368"  # @param ["vocals-bs-roformer-368", "vocals-bs-roformer-317", "vocals-bs-roformer-ft1-anvuew", "karaoke-bs-roformer-anvuew", "karaoke-mel-roformer-10.1956", "dereverb-bs-roformer-anvuew-22.5050", "dereverb-anvuew-19.1729", "dereverb-less-aggressive-18.8050"]
DEVICE = "auto"  # @param ["auto", "cuda:0", "cpu"]
FORMAT = "wav"  # @param ["wav", "flac", "mp3", "m4a"]
import time

source = input_path(INPUT)
target = OUTPUT_DIR / "separate" / PRESET
target.mkdir(parents=True, exist_ok=True)
started = time.time() - 1
rvc("separate", "--input", source, "--output", target, "--preset", PRESET, "--device", DEVICE,
    "--format", FORMAT, "--overwrite")
show_new(target, started)


In [ ]:
# @title E · Обучение голоса
#@markdown Датасет: папка с чистым вокалом в /content/nvc-input или полный путь (можно на Drive).
DATASET = ""  # @param {type:"string"}
EXPERIMENT = "my-voice"  # @param {type:"string"}
PRETRAIN = "NVC-Pretrained"  # @param ["NVC-Pretrained", "RVC v2", "custom", "none"]
PRETRAINED_G = ""  # @param {type:"string"}
PRETRAINED_D = ""  # @param {type:"string"}
SAMPLE_RATE = 40000  # @param [32000, 40000, 48000] {type:"raw"}
EMBEDDER = "mhubert147"  # @param ["mhubert147", "hubert_base", "contentvec", "spin", "spin-v2"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "nmvpe", "pm", "fcpe"]
#@markdown Значения по умолчанию — рецепт, с которым проверялись голоса от NVC-Pretrained.
EPOCHS = 250  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"integer"}
SAVE_EVERY = 50  # @param {type:"integer"}
SAVE_WEIGHTS = True  # @param {type:"boolean"}
HF_BANDS = True  # @param {type:"boolean"}
UNVOICE_FRICATIVES = False  # @param {type:"boolean"}
#@markdown Многослойные входы: обучение на копии с даблами, гармониями и ревербом (для скримов и жёсткого вокала не включайте).
LAYER_AUG = False  # @param {type:"boolean"}
LAYER_AUG_PROB = 0.5  # @param {type:"slider", min:0.1, max:0.9, step:0.05}
LAYER_FX = False  # @param {type:"boolean"}
BUILD_INDEX = True  # @param {type:"boolean"}
FRESH = False  # @param {type:"boolean"}
EMA_DECAY = 0.0  # @param {type:"number"}
import os

if PRETRAIN == "NVC-Pretrained" and (SAMPLE_RATE != 40000 or EMBEDDER != "mhubert147"):
    print("NVC-Pretrained обучен на 40 кГц с признаками mHuBERT-147: ставлю частоту 40000 и эмбеддер mhubert147.")
    SAMPLE_RATE, EMBEDDER = 40000, "mhubert147"
experiment = EXPERIMENT.strip()
if not experiment:
    raise ValueError("EXPERIMENT")
folder = "logs/" + experiment
if DATASET.strip():
    rvc("preprocess", "--input", input_path(DATASET), "--experiment", folder,
        "--sample-rate", SAMPLE_RATE, "--workers", os.cpu_count() or 2)
extract = ["--f0-method", F0_METHOD, "--embedder", EMBEDDER, "--device", "cuda:0"]
if UNVOICE_FRICATIVES:
    extract.append("--unvoice-fricatives")
rvc("extract", "--experiment", folder, *extract)
train = ["train", "--experiment", folder, "--name", experiment, "--sample-rate", SAMPLE_RATE, "--embedder", EMBEDDER,
         "--epochs", EPOCHS, "--save-every", SAVE_EVERY, "--device", "cuda:0", "--ema-decay", EMA_DECAY]
if LAYER_AUG:
    layered = folder + "_layers"
    rvc("layer-aug", "--experiment", folder, "--output", layered, "--sample-rate", SAMPLE_RATE,
        *(["--fx"] if LAYER_FX else []))
    rvc("extract", "--experiment", layered, *extract)
    train += ["--layer-aug", layered, "--layer-aug-prob", LAYER_AUG_PROB]
rate = SAMPLE_RATE // 1000
if PRETRAIN == "NVC-Pretrained":
    train += ["--pretrained-g", "assets/pretrained_v2/NVC-Pretrained/f0G40k.pth",
              "--pretrained-d", "assets/pretrained_v2/NVC-Pretrained/f0D40k.pth"]
elif PRETRAIN == "custom":
    train += ["--pretrained-g", PRETRAINED_G.strip(), "--pretrained-d", PRETRAINED_D.strip()]
elif PRETRAIN == "none":
    train.append("--random-weights")
if BATCH_SIZE > 0:
    train += ["--batch-size", BATCH_SIZE]
if SAVE_WEIGHTS:
    train.append("--save-weights")
if HF_BANDS:
    train.append("--hf-bands")
if FRESH:
    train.append("--fresh")
rvc(*train)
if BUILD_INDEX:
    rvc("train-index", "--experiment", folder, "--workers", os.cpu_count() or 2)
print(WEIGHTS / (experiment + ".pth"))


In [ ]:
# @title F · Слои и эффекты
#@markdown Создаёт многослойную копию эксперимента для обучения (ячейка E делает это сама при LAYER_AUG).
EXPERIMENT = ""  # @param {type:"string"}
OUTPUT = ""  # @param {type:"string"}
SAMPLE_RATE = 40000  # @param [32000, 40000, 48000] {type:"raw"}
LAYERS = True  # @param {type:"boolean"}
FX = False  # @param {type:"boolean"}
#@markdown Вокал (файл или папка), к среднему спектру которого эффекты подтягивают клипы. Пусто: без профиля.
FX_PROFILE_FROM = ""  # @param {type:"string"}
#@markdown Папка с остатками разделения и минусами inst*, которые подмешиваются под входы. Пусто: без них.
BLEED = ""  # @param {type:"string"}
SEED = 0  # @param {type:"integer"}
EXTRACT_AFTER = True  # @param {type:"boolean"}
EMBEDDER = "mhubert147"  # @param ["mhubert147", "hubert_base", "contentvec", "spin", "spin-v2"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "nmvpe", "pm", "fcpe"]

output = "logs/" + (OUTPUT.strip() or EXPERIMENT.strip() + "_layers")
args = ["layer-aug", "--experiment", "logs/" + EXPERIMENT.strip(), "--output", output, "--sample-rate", SAMPLE_RATE, "--seed", SEED]
if not LAYERS:
    args.append("--no-layers")
if FX:
    args.append("--fx")
if FX_PROFILE_FROM.strip():
    profile = PROJECT / output / "fx_profile.json"
    profile.parent.mkdir(parents=True, exist_ok=True)
    rvc("fx-profile", "--input", input_path(FX_PROFILE_FROM), "--output", profile)
    args += ["--fx-profile", profile]
if BLEED.strip():
    args += ["--bleed", input_path(BLEED)]
rvc(*args)
if EXTRACT_AFTER:
    rvc("extract", "--experiment", output, "--f0-method", F0_METHOD, "--embedder", EMBEDDER, "--device", "cuda:0", "--unvoice-fricatives")


In [ ]:
# @title G · Работа с моделями
ACTION = "info"  # @param ["info", "describe", "merge", "export", "list"]
#@markdown info / describe / merge: имена моделей из assets/weights. export: чекпоинт G_*.pth из logs/<эксперимент>.
MODEL = ""  # @param {type:"string"}
SECOND_MODEL = ""  # @param {type:"string"}
ALPHA = 0.5  # @param {type:"slider", min:0, max:1, step:0.05}
DESCRIPTION = ""  # @param {type:"string"}
OUTPUT_NAME = ""  # @param {type:"string"}
SAMPLE_RATE = 40000  # @param [32000, 40000, 48000] {type:"raw"}
from pathlib import Path

def output_model(default):
    name = OUTPUT_NAME.strip() or default
    return WEIGHTS / (name if name.endswith(".pth") else name + ".pth")

if ACTION == "list":
    print("\n".join(models()) or "-")
elif ACTION == "info":
    rvc("model-info", model_path(MODEL))
elif ACTION == "describe":
    model = model_path(MODEL)
    rvc("set-model-info", model, output_model(model.stem + "_described"), "--description", DESCRIPTION, "--overwrite")
elif ACTION == "merge":
    first, second = model_path(MODEL), model_path(SECOND_MODEL)
    rvc("merge-models", first, second, output_model(first.stem + "_" + second.stem), "--alpha", ALPHA,
        "--description", DESCRIPTION, "--overwrite")
elif ACTION == "export":
    checkpoint = Path(MODEL.strip())
    if not checkpoint.is_absolute():
        checkpoint = PROJECT / "logs" / checkpoint
    rvc("export-model", checkpoint, output_model(checkpoint.parent.name + "_" + checkpoint.stem),
        "--sample-rate", SAMPLE_RATE, "--description", DESCRIPTION, "--overwrite")


In [ ]:
# @title H · Скачать результаты
INCLUDE_TRAINED_VOICES = False  # @param {type:"boolean"}
import shutil, time
from google.colab import files

bundle = Path("/content/nvc-results")
if bundle.exists():
    shutil.rmtree(bundle)
shutil.copytree(OUTPUT_DIR, bundle / "output")
if INCLUDE_TRAINED_VOICES:
    for path in list(WEIGHTS.glob("*.pth")) + list(INDICES.glob("*.index")):
        if not path.is_symlink():
            (bundle / "voices").mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, bundle / "voices" / path.name)
    for path in (PROJECT / "logs").glob("*/*added_*.index"):
        (bundle / "voices").mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, bundle / "voices" / path.name)
archive = shutil.make_archive("/content/nvc-results-" + time.strftime("%Y%m%d-%H%M%S"), "zip", bundle)
print("Архив" + ":", archive)
files.download(archive)
